## Adding Functions and Tools to Agents

In [1]:
import os
import asyncio
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

In [2]:
from dotenv import load_dotenv

load_dotenv('.env')
OPENAI_API_ENDPOINT = os.getenv("OPENAI_API_ENDPOINT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

### Using function tools
Function tools are just custom functions that the agent can call. You can pass python functions to an agent using the `tools` parameter.

You can provide additional descriptions about the function or its parameters to the agent using Python's type annotations with Annotated and Pydantic's Field to provide descriptions. This is so that the agent can more accurately choose between different functions.

In [69]:
# Creating a simple get_weather function, with type annotations for the agent to understand:
from typing import Annotated
from pydantic import Field
import requests

def get_weather(
    lat_lon: Annotated[list[float, float],
             Field(description="The latitude and longitude of the location to get the weather for.")]
) -> str:
    """Get the weather for a given lattitude and longitude."""
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_lon[0]}&longitude={lat_lon[1]}&current=temperature_2m"
    response = requests.get(url)
    data = response.json()
    temp = data['current']['temperature_2m']
    return f'The current temperature is {temp}°C.'

In [71]:
# Now we can create an agent with the get_weather function as a tool:
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

agent = AzureOpenAIChatClient(deployment_name="gpt-5-mini", endpoint=OPENAI_API_ENDPOINT, api_key=OPENAI_API_KEY
    ).create_agent(
        instructions="You are a helpful assistant",
        tools=[get_weather]
)

In [ ]:
# Try running the agent (note that if it doesn't have enough context, it won't be able to use the tool):
async def main():
    result = await agent.run("What is the weather like currently?")
    print(result.text)

await main()

I don’t know your location. Please tell me where you want the current weather for, or let me access your device location. You can give:

- a city name (e.g., "Seattle, WA"),
- a ZIP/postal code,
- or latitude/longitude (e.g., "47.61, -122.33").

Also tell me if you want metric (°C, mm) or imperial (°F, inches) units and whether you only want current conditions or a short forecast.


In [72]:
# Try running the agent again, this time with lat/lon context so it can use the tool:
async def main():
    result = await agent.run("What is the weather currently for latitude 40.7282 and longitude -74.0776?")
    print(result.text)

await main()

The current temperature at 40.7282, -74.0776 is 17.9°C (about 64.2°F). Would you like humidity, wind, a short-term forecast, or the temperature in a different unit?


In [ ]:
# We can also create a class with multiple tools for agents to use:
class WeatherTools():
    """ A collection of weather-related tools for agent to use."""
    def __init__(self):
        self.last_location = None

    def get_current_temperature(
        self,
        lat_lon: Annotated[list[float, float],
                 Field(description="The latitude and longitude of the location to get the temperature for.")]
    ) -> str:
        """ Get the current temperature for a given latitude and longitude. """
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_lon[0]}&longitude={lat_lon[1]}&current=temperature_2m"
        response = requests.get(url)
        data = response.json()
        temp = data['current']['temperature_2m']
        return f'The current temperature is {temp}°C.'
    
    def get_precipitation_status(
        self,
        lat_lon: Annotated[list[float, float],
                 Field(description="The latitude and longitude of the location to get the precipitation status for.")]
    ) -> str:
        """ Get the current precipitation status for a given latitude and longitude. """
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_lon[0]}&longitude={lat_lon[1]}&current=precipitation"
        response = requests.get(url)
        data = response.json()
        precip = data['current']['precipitation']
        return f'There has been {precip} mm of precipitation in the past 15 minutes.'
    
    def get_humidity(
        self,
        lat_lon: Annotated[list[float, float],
                 Field(description="The latitude and longitude of the location to get the humidity for.")]
    ) -> str:
        """ Get the current humidity for a given latitude and longitude. """
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_lon[0]}&longitude={lat_lon[1]}&current=relative_humidity_2m"
        response = requests.get(url)
        data = response.json()
        humidity = data['current']['relative_humidity_2m']
        return f'The current relative humidity is {humidity}% at 2 meters above ground level.'

weather_agent = AzureOpenAIChatClient(deployment_name="gpt-5-mini", endpoint=OPENAI_API_ENDPOINT, api_key=OPENAI_API_KEY
    ).create_agent(
        name="WeatherAgent",
        description="An agent that can provide weather information using various tools.",
        instructions="You are a helpful assistant with tools to get weather information.",
        tools=[WeatherTools().get_current_temperature, WeatherTools().get_precipitation_status, WeatherTools().get_humidity]
)

# Try running the weather agent with multiple tools:
thread = weather_agent.get_new_thread()
async def main():
    result = await weather_agent.run("What is the current temperature in Amsterdam?", thread=thread)
    print(result.text)

    result = await weather_agent.run("How about the humidity?", thread=thread)
    print(result.text)

await main()

The current temperature in Amsterdam is 13.2°C (about 55.8°F). Would you like humidity or precipitation info as well?
The current relative humidity in Amsterdam is 98% (measured at 2 m). That’s very high — the air is nearly saturated, so fog, mist, or surface condensation is likely (the dew point is essentially the same as the air temperature). Would you like the current precipitation or visibility too?


### Agents as function tools
Defining agents as function tools, so that one agent can call another.

In [86]:
# We can create another agent that uses the weather agent as a tool. Note how we use the `as_tool()` method to
# convert the agent into a tool:
always_french_agent = AzureOpenAIChatClient(deployment_name="gpt-5-mini", endpoint=OPENAI_API_ENDPOINT, api_key=OPENAI_API_KEY
    ).create_agent(
    name="AlwaysFrenchAgent",
    instructions="You are a helpful assistant who only responds in French.",
    tools=weather_agent.as_tool()
)

result = await always_french_agent.run("What is the weather like in Paris?")
print(result.text)

Actuellement à Paris :
- Température : 19,2 °C
- Précipitations (dernières 15 min) : 0,0 mm — pas de pluie
- Humidité relative : 85 % — assez humide

Souhaitez-vous une prévision à court terme, les informations sur le vent, ou un bulletin horaire ?
